<a href="https://colab.research.google.com/github/srilakshmi-saladi/unet/blob/main/DRGrading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

# CHANGE THIS to where your zip is in Drive:
ZIP_PATH = "/content/drive/MyDrive/fundusDatasets/DR Grading/B_Disease Grading.zip"

OUT_DIR = "/content/idrid_grading"
os.makedirs(OUT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(OUT_DIR)

print("Unzipped to:", OUT_DIR)

# Show a quick tree
import glob
print("Some files:", glob.glob(OUT_DIR + "/**/*", recursive=True)[:30])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unzipped to: /content/idrid_grading
Some files: ['/content/idrid_grading/B. Disease Grading', '/content/idrid_grading/B. Disease Grading/2. Groundtruths', '/content/idrid_grading/B. Disease Grading/1. Original Images', '/content/idrid_grading/B. Disease Grading/CC-BY-4.0.txt', '/content/idrid_grading/B. Disease Grading/LICENSE.txt', '/content/idrid_grading/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv', '/content/idrid_grading/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv', '/content/idrid_grading/B. Disease Grading/1. Original Images/b. Testing Set', '/content/idrid_grading/B. Disease Grading/1. Original Images/a. Training Set', '/content/idrid_grading/B. Disease Grading/1. Original Images/b. Testing Set/IDRiD_063.jpg', '/content/idrid_grading/B. Disease Grading/1. Original Images/b. Testing 

In [3]:
import glob

ROOT = "/content/idrid_grading/B. Disease Grading"

csvs = glob.glob(ROOT + "/**/*.csv", recursive=True)
print("Found CSVs:")
for c in csvs:
    print(" -", c)

print("\nJPG counts:")
train_imgs = glob.glob(ROOT + "/1. Original Images/a. Training Set/*.jpg")
test_imgs  = glob.glob(ROOT + "/1. Original Images/b. Testing Set/*.jpg")
print("Train images:", len(train_imgs))
print("Test images :", len(test_imgs))
print("Example train img:", train_imgs[:1])
print("Example test img :", test_imgs[:1])

Found CSVs:
 - /content/idrid_grading/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
 - /content/idrid_grading/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv

JPG counts:
Train images: 413
Test images : 103
Example train img: ['/content/idrid_grading/B. Disease Grading/1. Original Images/a. Training Set/IDRiD_402.jpg']
Example test img : ['/content/idrid_grading/B. Disease Grading/1. Original Images/b. Testing Set/IDRiD_063.jpg']


In [4]:
import pandas as pd
import os

# Pick the first CSV by default (we will adjust if multiple)
if len(csvs) == 0:
    raise ValueError("No CSV found in Groundtruths. Check unzip path.")
CSV_PATH = csvs[0]
print("Using CSV:", CSV_PATH)

df = pd.read_csv(CSV_PATH)
print("\nColumns:", df.columns.tolist())
print(df.head())

# ---- auto-detect image column ----
possible_image_cols = ["Image name", "Image", "image", "img", "filename", "File name"]
image_col = None
for c in possible_image_cols:
    if c in df.columns:
        image_col = c
        break
if image_col is None:
    # fallback: first column
    image_col = df.columns[0]

# ---- auto-detect DR grade column ----
possible_grade_cols = ["Retinopathy grade", "DR grade", "grade", "dr_grade", "retinopathy"]
grade_col = None
for c in possible_grade_cols:
    if c in df.columns:
        grade_col = c
        break

if grade_col is None:
    # fallback: try any column containing "grade"
    for c in df.columns:
        if "grade" in c.lower():
            grade_col = c
            break

if grade_col is None:
    raise ValueError("Could not auto-detect DR grade column. Print df.columns and set grade_col manually.")

print("\nDetected image_col:", image_col)
print("Detected grade_col:", grade_col)

# Normalize filename (ensure .jpg)
def ensure_ext(x):
    x = str(x)
    if x.lower().endswith((".jpg", ".jpeg", ".png")):
        return x
    return x + ".jpg"

df["filename"] = df[image_col].apply(ensure_ext)
df["label"] = df[grade_col].astype(int)

print("\nLabel distribution:")
print(df["label"].value_counts().sort_index())

Using CSV: /content/idrid_grading/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv

Columns: ['Image name', 'Retinopathy grade', 'Risk of macular edema ', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']
  Image name  Retinopathy grade  Risk of macular edema   Unnamed: 3  \
0  IDRiD_001                  3                       2         NaN   
1  IDRiD_002                  3                       2         NaN   
2  IDRiD_003                  2                       2         NaN   
3  IDRiD_004                  3                       2         NaN   
4  IDRiD_005                  4                       0         NaN   

   Unnamed: 4  Unnamed: 5  Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  \
0         NaN         NaN         NaN         NaN         NaN         NaN   
1         NaN         NaN         NaN         NaN         NaN         NaN   
2         NaN         NaN   

In [5]:
TRAIN_IMG_DIR = ROOT + "/1. Original Images/a. Training Set"
TEST_IMG_DIR  = ROOT + "/1. Original Images/b. Testing Set"

df["path"] = df["filename"].apply(lambda x: os.path.join(TRAIN_IMG_DIR, x))
before = len(df)
df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
after = len(df)

print("Rows in CSV:", before)
print("Matched labeled TRAIN images:", after)
print("Example row:\n", df.head(1)[["filename","label","path"]])

Rows in CSV: 413
Matched labeled TRAIN images: 413
Example row:
         filename  label                                               path
0  IDRiD_001.jpg      3  /content/idrid_grading/B. Disease Grading/1. O...


In [15]:
import os, glob

ROOT = "/content/idrid_grading/B. Disease Grading"
csvs = sorted(glob.glob(ROOT + "/2. Groundtruths/**/*.csv", recursive=True))

print("All Groundtruth CSVs found:")
for i, c in enumerate(csvs):
    print(i, "->", os.path.basename(c), "|", c)

All Groundtruth CSVs found:
0 -> a. IDRiD_Disease Grading_Training Labels.csv | /content/idrid_grading/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
1 -> b. IDRiD_Disease Grading_Testing Labels.csv | /content/idrid_grading/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv


In [6]:
import pandas as pd

def pick_train_csv(csv_list):
    # prefer filenames containing 'train'
    for c in csv_list:
        if "train" in os.path.basename(c).lower():
            return c
    # else prefer bigger CSV (training is larger than testing)
    sizes = [(c, len(pd.read_csv(c))) for c in csv_list]
    sizes = sorted(sizes, key=lambda x: x[1], reverse=True)
    return sizes[0][0]

TRAIN_CSV = pick_train_csv(csvs)
print("Selected TRAIN_CSV:", TRAIN_CSV)

train_labels = pd.read_csv(TRAIN_CSV)
print("Rows in TRAIN_CSV:", len(train_labels))
print(train_labels.head())
print("Columns:", train_labels.columns.tolist())


Selected TRAIN_CSV: /content/idrid_grading/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
Rows in TRAIN_CSV: 413
  Image name  Retinopathy grade  Risk of macular edema   Unnamed: 3  \
0  IDRiD_001                  3                       2         NaN   
1  IDRiD_002                  3                       2         NaN   
2  IDRiD_003                  2                       2         NaN   
3  IDRiD_004                  3                       2         NaN   
4  IDRiD_005                  4                       0         NaN   

   Unnamed: 4  Unnamed: 5  Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  \
0         NaN         NaN         NaN         NaN         NaN         NaN   
1         NaN         NaN         NaN         NaN         NaN         NaN   
2         NaN         NaN         NaN         NaN         NaN         NaN   
3         NaN         NaN         NaN         NaN         NaN         NaN   
4         NaN         NaN         NaN     

In [7]:
import numpy as np

# Detect columns again
possible_image_cols = ["Image name", "Image", "image", "img", "filename", "File name"]
possible_grade_cols = ["Retinopathy grade", "DR grade", "grade", "dr_grade", "retinopathy"]

image_col = next((c for c in possible_image_cols if c in train_labels.columns), train_labels.columns[0])
grade_col = next((c for c in possible_grade_cols if c in train_labels.columns), None)

if grade_col is None:
    for c in train_labels.columns:
        if "grade" in c.lower():
            grade_col = c
            break
if grade_col is None:
    raise ValueError("Could not detect grade column; set grade_col manually.")

print("Using image_col:", image_col)
print("Using grade_col:", grade_col)

def ensure_ext(x):
    x = str(x).strip()
    if x.lower().endswith((".jpg",".jpeg",".png")):
        return x
    return x + ".jpg"

TRAIN_IMG_DIR = ROOT + "/1. Original Images/a. Training Set"

df = train_labels.copy()
df["filename"] = df[image_col].apply(ensure_ext)
df["label"] = df[grade_col].astype(int)
df["path"] = df["filename"].apply(lambda x: os.path.join(TRAIN_IMG_DIR, x))

before = len(df)
df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
after = len(df)

print("Rows in CSV:", before)
print("Matched TRAIN images:", after)
print("Label distribution:\n", df["label"].value_counts().sort_index())
print("Example:\n", df.head(1)[["filename","label","path"]])

Using image_col: Image name
Using grade_col: Retinopathy grade
Rows in CSV: 413
Matched TRAIN images: 413
Label distribution:
 label
0    134
1     20
2    136
3     74
4     49
Name: count, dtype: int64
Example:
         filename  label                                               path
0  IDRiD_001.jpg      3  /content/idrid_grading/B. Disease Grading/1. O...


In [8]:
from sklearn.model_selection import train_test_split

SEED = 42
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["label"]
)

print("Train:", len(train_df), "Val:", len(val_df))
print("Train label dist:\n", train_df["label"].value_counts().sort_index())
print("Val label dist:\n", val_df["label"].value_counts().sort_index())

Train: 330 Val: 83
Train label dist:
 label
0    107
1     16
2    109
3     59
4     39
Name: count, dtype: int64
Val label dist:
 label
0    27
1     4
2    27
3    15
4    10
Name: count, dtype: int64


In [10]:
!pip -q install opencv-python scikit-learn

import cv2, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20
NUM_CLASSES = 5
MODEL_NAME = "efficientnetb3"  # change to: "resnet50", "densenet121", "efficientnetb0"

AUTOTUNE = tf.data.AUTOTUNE

def crop_black_border(img_bgr, tol=7):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mask = gray > tol
    if mask.any():
        coords = np.argwhere(mask)
        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0) + 1
        return img_bgr[y0:y1, x0:x1]
    return img_bgr

def load_and_preprocess_opencv(path, img_size):
    img = cv2.imread(path)
    img = crop_black_border(img, tol=7)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32)

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="augment")

def tf_load_preprocess(path, label):
    def _pyfn(p):
        p = p.numpy().decode("utf-8")
        img = load_and_preprocess_opencv(p, IMG_SIZE)
        return img
    img = tf.py_function(_pyfn, [path], Tout=tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    img = img / 255.0
    y = tf.one_hot(label, NUM_CLASSES)
    return img, y

def make_ds(dataframe, training=True):
    paths = dataframe["path"].values
    labels = dataframe["label"].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(tf_load_preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df, training=False)

def build_model(model_name):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    if model_name == "efficientnetb0":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet")
        preprocess = keras.applications.efficientnet.preprocess_input
    elif model_name == "efficientnetb3":
        base = keras.applications.EfficientNetB3(include_top=False, weights="imagenet")
        preprocess = keras.applications.efficientnet.preprocess_input
    elif model_name == "resnet50":
        base = keras.applications.ResNet50(include_top=False, weights="imagenet")
        preprocess = keras.applications.resnet.preprocess_input
    elif model_name == "densenet121":
        base = keras.applications.DenseNet121(include_top=False, weights="imagenet")
        preprocess = keras.applications.densenet.preprocess_input
    else:
        raise ValueError("Unknown model_name")

    x = preprocess(inputs * 255.0)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    return model, base

model, base = build_model(MODEL_NAME)
model.summary()

# class weights
classes = np.arange(NUM_CLASSES)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["label"].values)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print("Class weights:", class_weight)

callbacks = [
    keras.callbacks.ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

# Stage A: head training
base.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

print("\n--- Stage A: Train head ---")
model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=class_weight, callbacks=callbacks)

# Stage B: fine-tune
base.trainable = True
N = 40
for layer in base.layers[:-N]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-4),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

print("\n--- Stage B: Fine-tune ---")
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=class_weight, callbacks=callbacks)

model = keras.models.load_model("best_model.keras")

# Evaluate
y_true, y_pred = [], []
for x, y in val_ds:
    p = model.predict(x, verbose=0)
    y_true.extend(np.argmax(y.numpy(), axis=1))
    y_pred.extend(np.argmax(p, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

acc = (y_true == y_pred).mean()
qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

print("\nVAL Accuracy:", acc)
print("VAL QWK:", qwk)
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, digits=4))

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply (Multiply)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 7, 7, 1536)     │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         7,685 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,791,220 (41.17 MB)

 Trainable params: 10,703,917 (40.83 MB)

 Non-trainable params: 87,303 (341.03 KB)

Class weights: {0: 0.616822429906542, 1: 4.125, 2: 0.6055045871559633, 3: 1.11864406779661, 4: 1.6923076923076923}

--- Stage A: Train head ---
Epoch 1/8
21/21 ━━━━━━━━━━━━━━━━━━━━ 198s 6s/step - accuracy: 0.3294 - loss: 1.5997 - val_accuracy: 0.3494 - val_loss: 1.4936 - learning_rate: 0.0010
Epoch 2/8
21/21 ━━━━━━━━━━━━━━━━━━━━ 104s 5s/step - accuracy: 0.3391 - loss: 1.3375 - val_accuracy: 0.4337 - val_loss: 1.3345 - learning_rate: 0.0010
Epoch 3/8
21/21 ━━━━━━━━━━━━━━━━━━━━ 122s 6s/step - accuracy: 0.3886 - loss: 1.4147 - val_accuracy: 0.4699 - val_loss: 1.2801 - learning_rate: 0.0010
Epoch 4/8
21/21 ━━━━━━━━━━━━━━━━━━━━ 103s 5s/step - accuracy: 0.4793 - loss: 1.2052 - val_accuracy: 0.4819 - val_loss: 1.2256 - learning_rate: 0.0010
Epoch 5/8
21/21 ━━━━━━━━━━━━━━━━━━━━ 122s 6s/step - accuracy: 0.4302 - loss: 1.2279 - val_accuracy: 0.5181 - val_loss: 1.1749 - learning_rate: 0.0010
Epoch 6/8
21/21 ━━━━━━━━━━━━━━━━━━━━ 101s 5s/step - accuracy: 0.4636 - loss: 1.0744 - val_accuracy: 0.4819

In [11]:
# ==========================================
# PRINT ACTUAL vs PREDICTED RESULTS
# ==========================================

import numpy as np
import pandas as pd

# ------------------------------------------
# 1️⃣ Recompute predictions on validation set
# ------------------------------------------

# Recreate validation dataset (important to avoid exhaustion)
val_ds_eval = make_ds(val_df, training=False)

y_true = []
y_pred = []
y_prob = []

for x_batch, y_batch in val_ds_eval:
    preds = model.predict(x_batch, verbose=0)

    # Store actual labels
    y_true.extend(np.argmax(y_batch.numpy(), axis=1))

    # Store predicted labels
    y_pred.extend(np.argmax(preds, axis=1))

    # Store prediction probabilities
    y_prob.extend(preds)

# Convert to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

# Get image filenames
image_names = val_df["filename"].values

print("Total validation samples:", len(y_true))


# ------------------------------------------
# 2️⃣ Print ALL predictions
# ------------------------------------------

print("\n=== ALL PREDICTIONS ===\n")

for i in range(len(y_true)):
    print(f"Image: {image_names[i]} | Actual: {y_true[i]} | Predicted: {y_pred[i]}")


# ------------------------------------------
# 3️⃣ Print ONLY WRONG predictions
# ------------------------------------------

print("\n=== WRONG PREDICTIONS ===\n")

for i in range(len(y_true)):
    if y_true[i] != y_pred[i]:
        print(f"Image: {image_names[i]} | Actual: {y_true[i]} | Predicted: {y_pred[i]}")


# ------------------------------------------
# 4️⃣ Print SEVERE ERRORS (Difference >= 2)
# ------------------------------------------

print("\n=== SEVERE ERRORS (|Actual - Predicted| >= 2) ===\n")

for i in range(len(y_true)):
    if abs(y_true[i] - y_pred[i]) >= 2:
        print(f"Image: {image_names[i]} | Actual: {y_true[i]} | Predicted: {y_pred[i]}")


# ------------------------------------------
# 5️⃣ Print with CONFIDENCE score
# ------------------------------------------

print("\n=== PREDICTIONS WITH CONFIDENCE ===\n")

for i in range(len(y_true)):
    confidence = np.max(y_prob[i])
    print(f"Image: {image_names[i]} | Actual: {y_true[i]} | Predicted: {y_pred[i]} | Confidence: {confidence:.4f}")


# ------------------------------------------
# 6️⃣ Save results to CSV file
# ------------------------------------------

results_df = pd.DataFrame({
    "Image": image_names,
    "Actual": y_true,
    "Predicted": y_pred,
    "Confidence": np.max(y_prob, axis=1),
    "Difference": np.abs(y_true - y_pred)
})

results_df.to_csv("validation_results.csv", index=False)

print("\nResults saved as validation_results.csv")

# Optional: download file
# from google.colab import files
# files.download("validation_results.csv")

Total validation samples: 83

=== ALL PREDICTIONS ===

Image: IDRiD_412.jpg | Actual: 2 | Predicted: 1
Image: IDRiD_124.jpg | Actual: 2 | Predicted: 2
Image: IDRiD_138.jpg | Actual: 0 | Predicted: 0
Image: IDRiD_298.jpg | Actual: 0 | Predicted: 0
Image: IDRiD_102.jpg | Actual: 2 | Predicted: 3
Image: IDRiD_158.jpg | Actual: 0 | Predicted: 0
Image: IDRiD_369.jpg | Actual: 3 | Predicted: 3
Image: IDRiD_391.jpg | Actual: 2 | Predicted: 2
Image: IDRiD_093.jpg | Actual: 2 | Predicted: 3
Image: IDRiD_378.jpg | Actual: 0 | Predicted: 0
Image: IDRiD_319.jpg | Actual: 2 | Predicted: 2
Image: IDRiD_070.jpg | Actual: 4 | Predicted: 0
Image: IDRiD_207.jpg | Actual: 3 | Predicted: 2
Image: IDRiD_408.jpg | Actual: 1 | Predicted: 0
Image: IDRiD_380.jpg | Actual: 3 | Predicted: 3
Image: IDRiD_098.jpg | Actual: 3 | Predicted: 3
Image: IDRiD_157.jpg | Actual: 0 | Predicted: 0
Image: IDRiD_119.jpg | Actual: 4 | Predicted: 4
Image: IDRiD_117.jpg | Actual: 2 | Predicted: 2
Image: IDRiD_337.jpg | Actual: 2 

In [ ]:
# ============================================================
# UPDATED: EfficientNetB3 IDRiD Training (Stronger Baseline)
# - Focal Loss
# - Save BEST by QWK
# - Freeze BatchNorm in fine-tuning
# - Cosine LR schedule
# ============================================================

!pip -q install opencv-python scikit-learn

import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score

# ----------------------------
# CONFIG
# ----------------------------
IMG_SIZE = 224        # try 384 later if GPU allows (better lesions)
BATCH_SIZE = 16
EPOCHS = 20
NUM_CLASSES = 5
MODEL_NAME = "efficientnetb3"
AUTOTUNE = tf.data.AUTOTUNE

# Make sure SEED exists; if not, set it here
try:
    SEED
except NameError:
    SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ----------------------------
# Preprocessing: crop black borders
# ----------------------------
def crop_black_border(img_bgr, tol=7):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mask = gray > tol
    if mask.any():
        coords = np.argwhere(mask)
        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0) + 1
        return img_bgr[y0:y1, x0:x1]
    return img_bgr

def load_and_preprocess_opencv(path, img_size):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Could not read image: {path}")
    img = crop_black_border(img, tol=7)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32)

# ----------------------------
# Augmentation (safe / mild)
# ----------------------------
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="augment")

# ----------------------------
# tf.data pipeline
# ----------------------------
def tf_load_preprocess(path, label):
    def _pyfn(p):
        p = p.numpy().decode("utf-8")
        img = load_and_preprocess_opencv(p, IMG_SIZE)
        return img

    img = tf.py_function(_pyfn, [path], Tout=tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])

    # Normalize to [0,1]
    img = img / 255.0
    y = tf.one_hot(label, NUM_CLASSES)
    return img, y

def make_ds(dataframe, training=True):
    paths = dataframe["path"].values
    labels = dataframe["label"].values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(tf_load_preprocess, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df, training=False)

# ----------------------------
# Model builder
# ----------------------------
def build_model(model_name):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    if model_name == "efficientnetb3":
        base = keras.applications.EfficientNetB3(include_top=False, weights="imagenet")
        preprocess = keras.applications.efficientnet.preprocess_input
    elif model_name == "efficientnetb0":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet")


In [9]:
# ============================================================
# EfficientNetB3 - QWK Focused Training (IDRiD)
# Target: push QWK toward > 0.85 (single model)
# Improvements:
#  - IMG_SIZE=384 (better lesion detail)
#  - Green-channel CLAHE preprocessing
#  - Balanced oversampling (helps rare grade 1)
#  - MixUp augmentation (strong regularization)
#  - Focal loss + label smoothing
#  - Freeze BatchNorm during fine-tuning
#  - Cosine LR schedule
#  - Save best checkpoint by QWK
# ============================================================

!pip -q install opencv-python scikit-learn

import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import cohen_kappa_score

# ----------------------------
# CONFIG
# ----------------------------
try:
    SEED
except NameError:
    SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

NUM_CLASSES = 5
BATCH_SIZE = 8          # 384 needs more memory. If you have a strong GPU, try 16.
IMG_SIZE = 384          # If OOM -> 299 or 224
EPOCHS_A = 10           # Stage A (head)
EPOCHS_B = 25           # Stage B (fine-tune)
FINE_TUNE_LAST_N = 60   # fine-tune more layers than before
AUTOTUNE = tf.data.AUTOTUNE

# ----------------------------
# Preprocessing: crop border + CLAHE (green channel)
# ----------------------------
def crop_black_border(img_bgr, tol=7):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mask = gray > tol
    if mask.any():
        coords = np.argwhere(mask)
        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0) + 1
        return img_bgr[y0:y1, x0:x1]
    return img_bgr

def apply_clahe_green(img_rgb):
    """
    Apply CLAHE to green channel only (common in fundus preprocessing).
    """
    g = img_rgb[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g2 = clahe.apply(g.astype(np.uint8))
    img2 = img_rgb.copy()
    img2[:, :, 1] = g2
    return img2

def load_and_preprocess_opencv(path, img_size):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Could not read image: {path}")
    img = crop_black_border(img, tol=7)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = apply_clahe_green(img)
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32)

# ----------------------------
# Mild geometric + photometric augmentation
# ----------------------------
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.07),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="augment")

# ----------------------------
# tf.data: load + preprocess
# ----------------------------
def tf_load_preprocess(path, label):
    def _pyfn(p):
        p = p.numpy().decode("utf-8")
        img = load_and_preprocess_opencv(p, IMG_SIZE)
        return img

    img = tf.py_function(_pyfn, [path], Tout=tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])

    img = img / 255.0
    y = tf.one_hot(label, NUM_CLASSES)
    return img, y

# ----------------------------
# MixUp (no extra libraries)
# ----------------------------
def mixup_batch(x, y, alpha=0.2):
    """
    MixUp within a batch:
      x' = lam*x + (1-lam)*x_shuffled
      y' = lam*y + (1-lam)*y_shuffled
    """
    batch_size = tf.shape(x)[0]
    # sample lambda from Beta(alpha, alpha)
    lam = tf.random.gamma([batch_size], alpha, beta=1.0)
    lam2 = tf.random.gamma([batch_size], alpha, beta=1.0)
    lam = lam / (lam + lam2)
    lam_x = tf.reshape(lam, [batch_size, 1, 1, 1])
    lam_y = tf.reshape(lam, [batch_size, 1])

    idx = tf.random.shuffle(tf.range(batch_size), seed=SEED)
    x2 = tf.gather(x, idx)
    y2 = tf.gather(y, idx)

    x_mix = x * lam_x + x2 * (1 - lam_x)
    y_mix = y * lam_y + y2 * (1 - lam_y)
    return x_mix, y_mix

# ----------------------------
# Balanced oversampling to help rare classes (esp grade 1)
# ----------------------------
def make_balanced_train_ds(df_train):
    """
    Oversample minority classes by repeating examples.
    Simple & effective on small datasets.
    """
    # split per class
    class_datasets = []
    counts = df_train["label"].value_counts().to_dict()
    max_count = max(counts.values())

    for c in range(NUM_CLASSES):
        sub = df_train[df_train["label"] == c]
        if len(sub) == 0:
            continue
        ds = tf.data.Dataset.from_tensor_slices((sub["path"].values, sub["label"].values))
        # repeat enough times to roughly match max_count
        reps = int(np.ceil(max_count / len(sub)))
        ds = ds.repeat(reps)
        class_datasets.append(ds)

    # sample uniformly from class datasets
    ds = tf.data.Dataset.sample_from_datasets(class_datasets, weights=[1.0]*len(class_datasets), seed=SEED)
    ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(tf_load_preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).map(mixup_batch, num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

def make_val_ds(df_val):
    ds = tf.data.Dataset.from_tensor_slices((df_val["path"].values, df_val["label"].values))
    ds = ds.map(tf_load_preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_balanced_train_ds(train_df)
val_ds   = make_val_ds(val_df)

# ----------------------------
# Build model
# ----------------------------
def build_effnetb3():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    base = keras.applications.EfficientNetB3(include_top=False, weights="imagenet")
    preprocess = keras.applications.efficientnet.preprocess_input

    x = preprocess(inputs * 255.0)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.45)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    return model, base

model, base = build_effnetb3()
model.summary()

# ----------------------------
# QWK callback (save best by QWK)
# ----------------------------
class QWKCallback(keras.callbacks.Callback):
    def __init__(self, val_ds, save_path="best_by_qwk.keras"):
        super().__init__()
        self.val_ds = val_ds
        self.save_path = save_path
        self.best_qwk = -1.0

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []
        for x_batch, y_batch in self.val_ds:
            p = self.model.predict(x_batch, verbose=0)
            y_true.extend(np.argmax(y_batch.numpy(), axis=1))
            y_pred.extend(np.argmax(p, axis=1))

        qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        print(f"\nval_QWK: {qwk:.6f}")

        if qwk > self.best_qwk:
            self.best_qwk = qwk
            self.model.save(self.save_path)
            print(f"✅ Saved best model: {self.save_path} (QWK={qwk:.6f})")

# ----------------------------
# Loss: Focal + Label Smoothing
# (We implement smoothing by smoothing one-hot targets in the dataset)
# ----------------------------
LABEL_SMOOTH = 0.05

def smooth_labels(y, smoothing=0.05):
    return y * (1.0 - smoothing) + smoothing / NUM_CLASSES

# apply label smoothing to both train and val streams
train_ds = train_ds.map(lambda x, y: (x, smooth_labels(y, LABEL_SMOOTH)), num_parallel_calls=AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (x, smooth_labels(y, LABEL_SMOOTH)), num_parallel_calls=AUTOTUNE)

loss_fn = tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0)

# ----------------------------
# Stage A: Train head (freeze backbone)
# ----------------------------
base.trainable = False
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=loss_fn,
    metrics=["accuracy"]
)

callbacks_a = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    QWKCallback(val_ds, save_path="best_by_qwk_stageA.keras"),
]

print("\n--- Stage A (Head) ---")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_A,
    callbacks=callbacks_a,
    steps_per_epoch=60,   # IMPORTANT: because oversampling repeats forever; control with steps
)

# ----------------------------
# Stage B: Fine-tune last layers + freeze BatchNorm
# ----------------------------
base.trainable = True

# fine-tune last N layers
for layer in base.layers[:-FINE_TUNE_LAST_N]:
    layer.trainable = False

# freeze BatchNorm (important!)
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# cosine LR schedule
steps_per_epoch = 60
total_steps = steps_per_epoch * EPOCHS_B
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=total_steps,
    alpha=1e-6
)

model.compile(
    optimizer=keras.optimizers.Adam(lr_schedule),
    loss=loss_fn,
    metrics=["accuracy"]
)

callbacks_b = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    QWKCallback(val_ds, save_path="best_by_qwk.keras"),
]

print("\n--- Stage B (Fine-tune) ---")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_B,
    callbacks=callbacks_b,
    steps_per_epoch=steps_per_epoch,
)

# ----------------------------
# Final evaluation (without smoothing for true QWK)
# ----------------------------
best_model = keras.models.load_model("best_by_qwk.keras")

# rebuild a clean val_ds for evaluation (no smoothing)
val_ds_eval = make_val_ds(val_df)

y_true, y_pred = [], []
for x_batch, y_batch in val_ds_eval:
    p = best_model.predict(x_batch, verbose=0)
    y_true.extend(np.argmax(y_batch.numpy(), axis=1))
    y_pred.extend(np.argmax(p, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
acc = (y_true == y_pred).mean()
severe = np.mean(np.abs(y_true - y_pred) >= 2)

print("\n===== FINAL =====")
print("VAL Accuracy:", acc)
print("VAL QWK:", qwk)
print("Severe error rate (|diff|>=2):", severe)
print("Saved best model:", "best_by_qwk.keras")

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply (Multiply)             │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 12, 12, 1536)   │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         7,685 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,791,220 (41.17 MB)

 Trainable params: 10,703,917 (40.83 MB)

 Non-trainable params: 87,303 (341.03 KB)

...


val_QWK: 0.759051
60/60 ━━━━━━━━━━━━━━━━━━━━ 103s 2s/step - accuracy: 0.8400 - loss: 0.0968 - val_accuracy: 0.6386 - val_loss: 0.1386
Epoch 21/25
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7910 - loss: 0.1065
val_QWK: 0.773551
60/60 ━━━━━━━━━━━━━━━━━━━━ 289s 5s/step - accuracy: 0.7912 - loss: 0.1065 - val_accuracy: 0.6747 - val_loss: 0.1281
Epoch 22/25
24/60 ━━━━━━━━━━━━━━━━━━━━ 1:14 2s/step - accuracy: 0.8661 - loss: 0.0939

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



val_QWK: 0.767911
60/60 ━━━━━━━━━━━━━━━━━━━━ 101s 2s/step - accuracy: 0.8454 - loss: 0.0953 - val_accuracy: 0.6867 - val_loss: 0.1294
Epoch 23/25
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7998 - loss: 0.0968
val_QWK: 0.784788
60/60 ━━━━━━━━━━━━━━━━━━━━ 285s 5s/step - accuracy: 0.7999 - loss: 0.0968 - val_accuracy: 0.7108 - val_loss: 0.1345
Epoch 24/25
24/60 ━━━━━━━━━━━━━━━━━━━━ 1:12 2s/step - accuracy: 0.8699 - loss: 0.1025

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



val_QWK: 0.816293
60/60 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step - accuracy: 0.8722 - loss: 0.0999 - val_accuracy: 0.6867 - val_loss: 0.1360

===== FINAL =====
VAL Accuracy: 0.5542168674698795
VAL QWK: 0.8264372150423366
Severe error rate (|diff|>=2): 0.0963855421686747
Saved best model: best_by_qwk.keras
